# Nino Detector Trainer

Fine-tunes **EfficientDet-Lite0** on your custom class list so the Nino app can recognize the objects a blind/low-vision user finds hard to see.

**How it works**
1. Picks a curated set of classes (edit `CLASSES` below).
2. Downloads labeled images for those classes from the **Open Images** dataset (no manual labeling).
3. Optionally merges in your own labeled photos (people/pets/custom objects).
4. Fine-tunes EfficientDet-Lite0, then exports `detect.tflite` + `labels.txt`.

**Time expectations (free Colab GPU)**
- Open Images download: 15–60 min (first run downloads the annotation index once).
- Training: ~30–90 min for a first run. Keep this tab open — Colab free sessions disconnect after ~90 min idle.

**Run order**: run cell 0 (switch to Python 3.11), then **Runtime → Restart session**, then *Run all*. It runs straight through — the custom-photo upload is OFF by default (set `SKIP_CUSTOM_DATA = False` to enable it).

In [ ]:
# --- 0. Switch Colab to Python 3.11 (required by mediapipe-model-maker) ---
!sudo apt-get update -y -qq
!if ! command -v python3.11 >/dev/null 2>&1; then sudo add-apt-repository -y ppa:deadsnakes/ppa && sudo apt-get update -y -qq; fi
!sudo apt-get install -y -qq python3.11 python3.11-distutils
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --set python3 /usr/bin/python3.11
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py && python3 /tmp/get-pip.py -q
!python3 --version
!python3 -m pip --version

## After running cell 0 above
Do **Runtime → Restart session**. This relaunches the kernel on Python 3.11 (the next cell verifies it).
Then click **Run all** — the whole notebook runs from top to bottom.

In [ ]:
import sys
if sys.version_info[:2] != (3, 11):
    raise SystemExit(
        f'Kernel is Python {sys.version_info.major}.{sys.version_info.minor} \u2014 '
        'run cell 0, then Runtime → Restart session, then Run all again.'
    )
print('Kernel Python', sys.version_info.major, sys.version_info.minor, '✔')

In [ ]:
!python3 -m pip install -q --upgrade pip
!python3 -m pip install -q mediapipe-model-maker
!python3 -m pip install -q fiftyone
!python3 -m pip install -q opencv-python-headless pandas

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
print("fiftyone", fo.__version__)
from mediapipe_model_maker import object_detector
import tensorflow as tf
print("tensorflow", tf.__version__)
print("mediapipe_model_maker object_detector OK")

## 1. Choose your classes

This is the label set the trained model will recognize. It mixes **navigation-critical** objects (stairs, doors, signs, handrails, puddles...) with **everyday objects** (person, animals, vehicles, furniture...).

✏️ **Edit this list freely** — add or remove class names. Names must match the Open Images dataset exactly (the next cell checks them for you). Common extras you can add: `Dog`, `Cat`, `Bird`, `Boat`, `Train`, `Aircraft`, `Sheep`, `Horse`, `Cattle`, `Flower`, `Fountain`, `Fireplace`, `Mug`, `Knife`, `Fork`, `Spoon`, `Bowl`, `Wine glass`, `Pizza`, `Banana`, `Apple`, `Sandwich`, `Broccoli`, `Carrot`, `Hot dog`, `Remote control`, `Microwave oven`, `Houseplant`.

In [ ]:
# ---- EDIT ME -------------------------------------------------
CLASSES = [
    # navigation / hazards
    "Person", "Stairs", "Door", "Window", "Glass", "Fence", "Wall",
    "Handrail", "Guard rail", "Puddle", "Ramp", "Curb",
    "Traffic sign", "Street sign", "Stop sign", "Traffic light",
    # people / animals
    "Dog", "Cat", "Bird", "Horse", "Cattle", "Sheep",
    # vehicles
    "Car", "Truck", "Bus", "Motorcycle", "Bicycle", "Train", "Boat", "Aircraft",
    # furniture / household
    "Chair", "Couch", "Table", "Bed", "Desk", "Shelf", "Furniture",
    "Television", "Computer", "Laptop", "Mobile phone", "Remote control",
    "Microwave oven", "Refrigerator", "Stove", "Sink", "Toilet", "Bathtub",
    "Clock", "Book", "Backpack", "Bottle", "Cup", "Plate", "Bowl",
    "Keyboard", "Mouse", "Scissors", "Teddy bear", "Hair dryer", "Toothbrush",
    "Umbrella", "Bench", "Tree", "Plant", "Houseplant", "Flower",
]

MAX_SAMPLES_PER_CLASS = 120   # images pulled per class (higher = better, slower)
VAL_SPLIT = 0.10              # fraction held out for validation
EPOCHS = 30                   # training passes (higher = better, slower)
BATCH_SIZE = 8
IMAGE_SIZE = 320
SEED = 42
SKIP_CUSTOM_DATA = True  # True = Open Images only (no upload prompt); set False to add your own labeled photos
# ---------------------------------------------------------------

In [ ]:
import pandas as pd

!curl -sL "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions.csv" -o /content/oid_classes.csv
df = pd.read_csv("/content/oid_classes.csv", names=["id", "name"])
valid = set(df["name"])
missing = [c for c in CLASSES if c not in valid]
print(f"{len(CLASSES)} classes requested")
if missing:
    print("MISSING / INVALID class names (fix CLASSES above):")
    print(missing)
else:
    print("All class names are valid.\n")
    print("Available sample pick (extra ideas):")
    print([c for c in sorted(valid) if c in (
        "Fire hydrant", "Flag", "Billboard", "Suitcase", "Surfboard", "Skateboard",
        "Wheelchair", "Crane", "Tractor", "Hedge", "Houseplant")])

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

sets = []
for c in CLASSES:
    try:
        ds = foz.load_zoo_dataset(
            "open-images-v7",
            split="train",
            label_types=["detections"],
            classes=[c],
            max_samples=MAX_SAMPLES_PER_CLASS,
            shuffle=True,
            seed=SEED,
            dataset_name=f"nino_{c.lower().replace(' ', '_')}",
        )
        if len(ds) == 0:
            print(f'WARN {c}: 0 samples \u2014 skipping')
        else:
            sets.append(ds)
            print(f'{c}: {len(ds)} samples')
    except Exception as e:
        print(f'SKIP {c}: {e}')

if not sets:
    raise RuntimeError('No class data could be downloaded \u2014 check class names and network, then rerun.')
dataset = sets[0].concat(sets[1:])

# dedupe images that were pulled for multiple classes (also avoids train/val leaks)
seen, keep = set(), []
for sample in dataset:
    if sample.filepath not in seen:
        seen.add(sample.filepath)
        keep.append(sample.id)
dataset = dataset.select(keep)
print(f"\nTotal unique images: {len(dataset)}")

## 2. (Optional) Your own labeled photos

If you want the model to recognize **your specific people, pets, or objects**, add them to `CLASSES` above and include labeled photos here.

**Labeling workflow** (free):
1. Take 50–100 photos of each object (varied angles/lighting).
2. Label them in a free tool like [makesense.ai](https://www.makesense.ai/) or [Roboflow](https://roboflow.com/), drawing boxes around each object.
3. Export as **PascalVOC** — you get a zip containing `Annotations/` and `JPEGImages/`.

Run the next two cells, upload that zip, and the images will be merged into the training set.

In [ ]:
import zipfile, os
from google.colab import files

custom_dir = "/content/custom_voc"
if SKIP_CUSTOM_DATA:
    print('SKIP_CUSTOM_DATA=True \u2014 no custom photos needed, continuing with Open Images only.')
else:
    os.makedirs(custom_dir, exist_ok=True)
    print('Upload your PascalVOC zip (Annotations + JPEGImages):')
    up = files.upload()
    if up:
        zpath = list(up.keys())[0]
        with zipfile.ZipFile(zpath) as z:
            z.extractall(custom_dir)
        print('Extracted to', custom_dir)
        print(os.listdir(custom_dir))

In [ ]:
import os
annot_dir = os.path.join(custom_dir, "Annotations")
if os.path.isdir(annot_dir):
    custom_ds = fo.Dataset.from_dir(
        custom_dir, dataset_type=fo.types.VOCDetectionDataset
    )
    custom_ds.name = "custom_data"
    custom_ds.persistent = False
    before = len(dataset)
    dataset = dataset.concat(custom_ds)
    print(f"Merged {len(dataset) - before} custom images (total {len(dataset)})")
else:
    print("No custom data found \u2014 continuing with Open Images only.")

In [ ]:
import random

ids = list(dataset.values("id"))
random.seed(SEED); random.shuffle(ids)
n_val = max(1, int(len(ids) * VAL_SPLIT))
val_ids, train_ids = set(ids[:n_val]), set(ids[n_val:])

train_view = dataset.select(train_ids)
val_view = dataset.select(val_ids)

train_dir, val_dir = "/content/voc_train", "/content/voc_val"
train_view.export(train_dir, dataset_type=fo.types.VOCDetectionDataset,
                  label_field="detections", classes=CLASSES)
val_view.export(val_dir, dataset_type=fo.types.VOCDetectionDataset,
                label_field="detections", classes=CLASSES)
print("Exported VOC train", train_dir, "| val", val_dir)

In [ ]:
from mediapipe_model_maker import object_detector

train_data = object_detector.Dataset.from_pascal_voc_folder(
    annotation_dir="/content/voc_train/Annotations",
    image_dir="/content/voc_train/JPEGImages",
)
val_data = object_detector.Dataset.from_pascal_voc_folder(
    annotation_dir="/content/voc_val/Annotations",
    image_dir="/content/voc_val/JPEGImages",
)
print("train samples:", len(train_data), "| val samples:", len(val_data))

In [ ]:
import tempfile

spec = object_detector.EfficientDetLite0Spec(
    model_dir=tempfile.mkdtemp(),
    image_size=IMAGE_SIZE,
)
try:
    spec.hparams.epochs = EPOCHS
    spec.hparams.batch_size = BATCH_SIZE
except AttributeError:
    spec.epochs = EPOCHS
    spec.batch_size = BATCH_SIZE

print("Training EfficientDet-Lite0 \u2026 (this is the long step)")
model = object_detector.EfficientDetLite0.create(
    train_data, model_spec=spec, validation_data=val_data
)

In [ ]:
model.evaluate(val_data)
model.export_to_tflite("/content/detect.tflite")
model.export_labels("/content/labels.txt")
print("Classes exported:", len(open("/content/labels.txt").read().splitlines()))
print(open("/content/labels.txt").read())

In [ ]:
!zip -r -j /content/nino_model.zip /content/detect.tflite /content/labels.txt
from google.colab import files
files.download("/content/nino_model.zip")
print("Downloaded. Send me the zip and I'll install it into the Nino app.")

## Next steps
- Send me the `nino_model.zip` → I install `detect.tflite` + `labels.txt` into the app, rebuild, redeploy.
- Want better results? Re-run with higher `MAX_SAMPLES_PER_CLASS` / `EPOCHS`, or add more of your own photos. Each retrain gives a stronger model (the "feed it more data" loop).
- Keep the free Colab session alive: click the ⚡ icon → *Connect* → *Change runtime type* → **GPU**. Run cells in order.